# Transfer Learning + Grad-CAM (2026 companion)
**Maintainer: Harpreet Singh**

This notebook demonstrates a reproducible transfer-learning workflow on CIFAR-10 using a pretrained MobileNetV2 feature extractor, followed by a simple Grad-CAM visualization. It downloads data and pretrained weights on first run. Runtime and accuracy depend on hardware and package versions.

In [ ]:
import os, random, platform
import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print({"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__})
print("Devices:", tf.config.list_physical_devices())

## 1. Load and prepare data
For a quick demonstration, use a small balanced subset of CIFAR-10. The validation subset is held out from training.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
class_names = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
# Keep the first 5,000 training examples to limit resource use in a demo.
x_train, y_train = x_train[:5000], y_train[:5000].reshape(-1)
y_test = y_test.reshape(-1)
x_train = tf.image.resize(x_train, (96, 96)).numpy().astype("float32")
x_test = tf.image.resize(x_test[:2000], (96, 96)).numpy().astype("float32")
y_test = y_test[:2000]
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(2000, seed=SEED).batch(32).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(32).prefetch(tf.data.AUTOTUNE)
print(x_train.shape, x_test.shape)

## 2. Build a pretrained feature extractor
MobileNetV2 expects pixels scaled to [-1, 1]. The base is frozen for the first training stage.

In [ ]:
augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.08),
], name="augmentation")
base = keras.applications.MobileNetV2(
    input_shape=(96, 96, 3), include_top=False, weights="imagenet"
)
base.trainable = False
inputs = keras.Input((96, 96, 3))
x = augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.25)(x)
outputs = keras.layers.Dense(10, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 3. Train with callbacks
The checkpoints and TensorBoard logs are written to `artifacts/`. These directories are ignored by Git.

In [ ]:
os.makedirs("artifacts", exist_ok=True)
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("artifacts/transfer_best.keras", monitor="val_loss", save_best_only=True),
    keras.callbacks.TensorBoard(log_dir="artifacts/tensorboard"),
]
history = model.fit(train_ds, validation_data=test_ds, epochs=5, callbacks=callbacks)
print("Evaluation:", model.evaluate(test_ds, return_dict=True))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.grid(True); plt.show()

## 4. Grad-CAM explainability
Grad-CAM is an approximate visual explanation, not proof that the model reasons correctly. This helper finds the final convolutional layer in MobileNetV2.

In [ ]:
def make_gradcam_heatmap(image_batch, full_model, conv_model):
    # The model is decomposed into a convolutional feature model and classifier head.
    with tf.GradientTape() as tape:
        conv_outputs = conv_model(image_batch, training=False)
        tape.watch(conv_outputs)
        # Re-run the classifier head from the pooled convolutional activations.
        pooled = full_model.get_layer("global_average_pooling2d")(conv_outputs)
        pooled = full_model.get_layer("dropout")(pooled, training=False)
        preds = full_model.get_layer("dense")(pooled)
        class_index = tf.argmax(preds[0])
        score = preds[:, class_index]
    grads = tape.gradient(score, conv_outputs)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_outputs[0] * weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    return heatmap / (tf.reduce_max(heatmap) + keras.backend.epsilon())

# Grad-CAM wiring can vary if layer names change between Keras versions.
# Inspect model.summary() and layer names before using on a customized architecture.
conv_model = keras.Model(base.input, base.get_layer("out_relu").output)
sample = tf.convert_to_tensor(x_test[:1])
try:
    heatmap = make_gradcam_heatmap(sample, model, conv_model).numpy()
    plt.figure(figsize=(5, 5)); plt.imshow(x_test[0].astype("uint8"))
    plt.imshow(tf.image.resize(heatmap[..., None], (96,96)).numpy().squeeze(), alpha=.4, cmap="jet")
    plt.axis("off"); plt.show()
except Exception as exc:
    print("Grad-CAM needs layer-name adjustment for this Keras build:", repr(exc))

## 5. Save a portable Keras model
Only save a model after reviewing validation metrics and class mappings.

In [ ]:
model.save("artifacts/cifar10_transfer.keras")
reloaded = keras.models.load_model("artifacts/cifar10_transfer.keras")
print("Reload check:", reloaded.predict(x_test[:1], verbose=0).shape)
print("Predicted class:", class_names[int(np.argmax(reloaded.predict(x_test[:1], verbose=0)[0]))])